In [376]:
# ============================================================
# CELL 1 — Load data into Colab
# ============================================================

import pandas as pd
import numpy as np
import sqlite3
import json
import math
from pathlib import Path
from google.colab import files

INPUT_FILE = "/content/assignment_data.csv"

# Check that the file exists
if not Path(INPUT_FILE).exists():
    raise FileNotFoundError(
        f"File not found: {INPUT_FILE}. "
        "Upload assignment_data.csv to the Colab session first."
    )

# Load the CSV
df = pd.read_csv(INPUT_FILE)

print(f"Loaded file: {INPUT_FILE}")
print(f"Rows: {len(df)}")
print(f"Columns: {len(df.columns)}")

# Top 5 data view for validation
display(df.head())

Loaded file: /content/assignment_data.csv
Rows: 1389
Columns: 36


,facility_id,category_name,manufacturername,facility_name,jpin,title,pvname,vendor_lead_time,inv_norm,safety_stock,...,open_po_cases,final_suggestion,final_days_of_inventory,final_cases_suggestion,final_value,final_tonnage,mov_check,mrp,cp,sales_band
0,FC-001,Baby Care,Colgate-Palmolive,FC-Alpha,SKU-00786,"Colgate Super Junior Toothbrush, Ultra Soft, 2...",Baby Toothbrush,5,26,4,...,1.0,0,0,0,0,0,NaN,155.90,129.75,NaN
1,FC-001,Baby Care,Dabur India Limited,FC-Alpha,SKU-00726,"Dabur Lal Tail, 50ml Box",Baby Oil,5,9,4,...,1.0,0,0,0,0,0,NaN,79.47,51.47,Band B
2,FC-001,Baby Care,Dabur India Limited,FC-Alpha,SKU-00846,"Dabur Lal Tail, 100ml Box",Baby Oil,5,9,4,...,0.0,0,0,0,0,0,NaN,149.74,109.19,Band C
3,FC-001,Beverages,Dabur India Limited,FC-Alpha,SKU-00352,"Real Koolerz Fruit Drink, Mango, 10Rs Tetra Pak",Fruit Drinks,5,14,4,...,25.0,0,0,0,0,0,NaN,8.84,6.94,Band C
4,FC-001,Beverages,Dabur India Limited,FC-Alpha,SKU-00725,"Real Fruit Power Drink, Guava, 180ml Tetra Pak",Fruit Drinks,5,14,4,...,7.0,0,0,0,0,0,NaN,16.85,16.79,Band C


In [377]:
# ============================================================
# CELL 2 — Configuration for ease of fetching data in the future
# ============================================================
START_DATE = pd.Timestamp("2026-03-16")

OUTPUT_COLUMNS = [
    "final_suggestion",
    "final_cases_suggestion",
    "final_value",
    "final_days_of_inventory",
    "final_tonnage",
    "mov_check",
]

NUMERIC_COLUMNS = [
    "vendor_lead_time",
    "inv_norm",
    "safety_stock",
    "current_vendor_mov",
    "max_allocated_space",
    "case_size",
    "current_inventory",
    "max_drr",
    "deadweight",
    "orderedquantity",
    "open_po_value",
    "open_po_cases",
    "cp",
    "mrp",
]

DB_FILE = "replenishment.db"

In [378]:
# ============================================================
# CELL 2 — Validation if required columns are there
# ============================================================
required = {
    "facility_id",
    "jpin",
    "vendor_id",
    "vendor_name",
    "vendor_lead_time",
    "inv_norm",
    "safety_stock",
    "current_vendor_mov",
    "minimum_order_criteria",
    "max_allocated_space",
    "case_size",
    "current_inventory",
    "inventory_breakup",
    "max_drr",
    "deadweight",
    "open_po_details",
    "orderedquantity",
    "open_po_value",
    "open_po_cases",
    "cp",
    "mrp",
}

missing = sorted(required - set(df.columns))

if missing:
    raise ValueError(f"Missing required columns: {missing}")

print("Validation successful.")
print(f"The {len(required)} required columns are present.")

Validation successful.
The 21 required columns are present.


In [379]:
# ============================================================
# CELL 3 — Parsing the JSON
# ============================================================

def parse_json(value):
    """
    Parse valid JSON strings.
    Null/blank values become empty dictionaries.
    """
    if pd.isna(value) or value in ("", "{}", None):
        return {}

    return json.loads(value)

In [380]:
# ============================================================
# CELL 4 — Validating the Parsed JSON
# ============================================================
df["inventory_breakup"] = df["inventory_breakup"].map(parse_json)
df["open_po_details"] = df["open_po_details"].map(parse_json)

print("JSON validation successful.")

JSON validation successful.


In [381]:
# ============================================================
# CELL 5 — Viewing the Parsed JSON
# ============================================================
df[["inventory_breakup", "open_po_details"]].head()

,inventory_breakup,open_po_details
0,"{'sellable': None, 'contingency': None, 'pendi...","{'open_po_1': {'promise_date': '2026-03-18', '..."
1,"{'sellable': 22.0, 'contingency': None, 'pendi...","{'open_po_1': {'promise_date': '2026-03-19', '..."
2,"{'sellable': 32.0, 'contingency': None, 'pendi...",{}
3,"{'sellable': None, 'contingency': None, 'pendi...","{'open_po_1': {'promise_date': '2026-03-19', '..."
4,"{'sellable': 452.0, 'contingency': 15.0, 'pend...","{'open_po_1': {'promise_date': None, 'orderedq..."


In [382]:
# ============================================================
# CELL 6 — Restore JSON columns to strings
# ============================================================
df["inventory_breakup"] = df["inventory_breakup"].map(json.dumps)
df["open_po_details"] = df["open_po_details"].map(json.dumps)
print("Success")

Success


In [383]:
# ============================================================
# CELL 7 — Convert Numeric column
# ============================================================
for col in NUMERIC_COLUMNS:
    df[col] = pd.to_numeric(
        df[col],
        errors="coerce"
    ).fillna(0)

print("Numeric conversion completed.")

Numeric conversion completed.


In [384]:
# ============================================================
# CELL 8 — Calculate current days of Inventory
# ============================================================
df["current_days_raw"] = np.where(
    df["max_drr"] > 0,
    df["current_inventory"] / df["max_drr"],
    0.0,
)

df[[
    "jpin",
    "current_inventory",
    "max_drr",
    "current_days_raw"
]].head()

,jpin,current_inventory,max_drr,current_days_raw
0,SKU-00786,0,0,0.000000
1,SKU-00726,22,6,3.666667
2,SKU-00846,32,2,16.000000
3,SKU-00352,0,118,0.000000
4,SKU-00725,467,35,13.342857


In [385]:
# ============================================================
# CELL 9 — Calculate Inventory Position
# ============================================================

df["inventory_position"] = (
    df["current_inventory"]
    + df["orderedquantity"]
)

df[[
    "jpin",
    "current_inventory",
    "orderedquantity",
    "inventory_position"
]].head()

,jpin,current_inventory,orderedquantity,inventory_position
0,SKU-00786,0,24,24
1,SKU-00726,22,120,142
2,SKU-00846,32,0,32
3,SKU-00352,0,1000,1000
4,SKU-00725,467,210,677


In [386]:
# ============================================================
# CELL 10 — Calculate Target Coverage
# ============================================================

df["target_days"] = (
    np.maximum(
        df["inv_norm"],
        df["vendor_lead_time"]
    )
    + df["safety_stock"]
)

df["target_units"] = (
    df["target_days"]
    * df["max_drr"]
)

df[[
    "jpin",
    "inv_norm",
    "vendor_lead_time",
    "safety_stock",
    "target_days",
    "target_units"
]].head()

,jpin,inv_norm,vendor_lead_time,safety_stock,target_days,target_units
0,SKU-00786,26,5,4,30,0
1,SKU-00726,9,5,4,13,78
2,SKU-00846,9,5,4,13,26
3,SKU-00352,14,5,4,18,2124
4,SKU-00725,14,5,4,18,630


In [387]:
# ============================================================
# CELL 11 — Calculate Raw replenishment
# ============================================================
df["raw_replenishment"] = np.maximum(
    0,
    df["target_units"]
    - df["inventory_position"]
)

df[[
    "jpin",
    "target_units",
    "inventory_position",
    "raw_replenishment"
]].head()

,jpin,target_units,inventory_position,raw_replenishment
0,SKU-00786,0,24,0
1,SKU-00726,78,142,0
2,SKU-00846,26,32,0
3,SKU-00352,2124,1000,1124
4,SKU-00725,630,677,0


In [388]:
# ============================================================
# CELL 12 — Calculate Physical capacity
# ============================================================

residual_capacity = np.maximum(
    0,
    df["max_allocated_space"]
    - df["inventory_position"]
)

case_size = np.where(
    df["case_size"] > 0,
    df["case_size"],
    1
)

max_order_cases = np.floor(
    residual_capacity / case_size
).astype(int)

desired_cases = np.ceil(
    df["raw_replenishment"] / case_size
).astype(int)

desired_cases = np.maximum(
    desired_cases,
    0
)

df["residual_capacity"] = residual_capacity
df["max_order_cases"] = max_order_cases
df["desired_cases"] = desired_cases

df[[
    "jpin",
    "max_allocated_space",
    "inventory_position",
    "residual_capacity",
    "case_size",
    "max_order_cases",
    "desired_cases"
]].head()

,jpin,max_allocated_space,inventory_position,residual_capacity,case_size,max_order_cases,desired_cases
0,SKU-00786,48,24,24,24,1,0
1,SKU-00726,240,142,98,120,0,0
2,SKU-00846,120,32,88,60,1,0
3,SKU-00352,800,1000,0,40,0,29
4,SKU-00725,600,677,0,30,0,0


In [389]:

# ============================================================
# CELL 13 — Calculate MOV Required Cases
# ============================================================

def mov_required_cases(row):

    if row["max_drr"] <= 0 or row["current_vendor_mov"] <= 0:
        return 0

    criteria = str(
        row["minimum_order_criteria"]
    ).upper()

    mov = row["current_vendor_mov"]

    cs = max(
        int(row["case_size"]),
        1
    )

    # VALUE MOV
    if criteria == "VALUE":

        if row["cp"] <= 0:
            return 10**9

        units = math.ceil(
            mov / row["cp"]
        )

        return math.ceil(
            units / cs
        )

    # CASE MOV
    if criteria == "CASES":
        return math.ceil(mov)

    # TONNAGE MOV
    if criteria == "TONNAGE":

        if row["deadweight"] <= 0:
            return 10**9

        units = math.ceil(
            mov / row["deadweight"]
        )

        return math.ceil(
            units / cs
        )

    return 0

In [390]:
# ============================================================
# CELL 14 — Apply MOV Calculation
# ============================================================

df["mov_required_cases"] = df.apply(
    mov_required_cases,
    axis=1
)

df[[
    "jpin",
    "minimum_order_criteria",
    "current_vendor_mov",
    "case_size",
    "mov_required_cases"
]].head()

,jpin,minimum_order_criteria,current_vendor_mov,case_size,mov_required_cases
0,SKU-00786,VALUE,120000,24,0
1,SKU-00726,VALUE,30000,120,5
2,SKU-00846,VALUE,30000,60,5
3,SKU-00352,VALUE,30000,40,109
4,SKU-00725,VALUE,30000,30,60


In [391]:
# ============================================================
# CELL 15 — Calculate Final cases suggestion
# ============================================================

df["final_cases_suggestion"] = np.minimum(
    df["max_order_cases"],
    np.maximum(
        df["desired_cases"],
        df["mov_required_cases"]
    )
).astype(int)

In [392]:
# ============================================================
# CELL 16 — Calculate Final unit suggestion
# ============================================================

df["final_suggestion"] = (
    df["final_cases_suggestion"]
    * df["case_size"]
).astype(int)

In [393]:
# ============================================================
# CELL 17 — Zero demand SKUs
# ============================================================

df.loc[
    df["max_drr"] <= 0,
    [
        "final_cases_suggestion",
        "final_suggestion"
    ]
] = 0

In [394]:
# ============================================================
# CELL 18 — Calculate final PO value
# ============================================================

df["final_value"] = (
    df["final_suggestion"]
    * df["cp"]
).round(2)

df[[
    "jpin",
    "final_suggestion",
    "cp",
    "final_value"
]].head()

,jpin,final_suggestion,cp,final_value
0,SKU-00786,0,129.75,0.0
1,SKU-00726,0,51.47,0.0
2,SKU-00846,60,109.19,6551.4
3,SKU-00352,0,6.94,0.0
4,SKU-00725,0,16.79,0.0


In [395]:
# ============================================================
# CELL 19 — Calculate Projected inventory
# ============================================================

df["projected_inventory"] = (
    df["current_inventory"]
    + df["orderedquantity"]
    + df["final_suggestion"]
)

df[[
    "jpin",
    "current_inventory",
    "orderedquantity",
    "final_suggestion",
    "projected_inventory"
]].head()

,jpin,current_inventory,orderedquantity,final_suggestion,projected_inventory
0,SKU-00786,0,24,0,24
1,SKU-00726,22,120,0,142
2,SKU-00846,32,0,60,92
3,SKU-00352,0,1000,0,1000
4,SKU-00725,467,210,0,677


In [396]:
# ============================================================
# CELL 20 — Calculate Final Days Of Inventory
# ============================================================
df["final_days_of_inventory"] = np.where(
    df["max_drr"] > 0,
    np.floor(
        df["projected_inventory"]
        / df["max_drr"]
    ),
    0,
).astype(int)

df[[
    "jpin",
    "max_drr",
    "projected_inventory",
    "final_days_of_inventory"
]].head()

,jpin,max_drr,projected_inventory,final_days_of_inventory
0,SKU-00786,0,24,0
1,SKU-00726,6,142,23
2,SKU-00846,2,92,46
3,SKU-00352,118,1000,8
4,SKU-00725,35,677,19


In [397]:
# ============================================================
# CELL 21 — Calculate Tonnage
# ============================================================

df["final_tonnage"] = (
    df["final_suggestion"]
    * df["deadweight"]
    / 1000
).round(4)

df[[
    "jpin",
    "final_suggestion",
    "deadweight",
    "final_tonnage"
]].head()

,jpin,final_suggestion,deadweight,final_tonnage
0,SKU-00786,0,0.000,0.0000
1,SKU-00726,0,0.068,0.0000
2,SKU-00846,60,0.121,0.0073
3,SKU-00352,0,0.167,0.0000
4,SKU-00725,0,0.200,0.0000


In [398]:
# ============================================================
# CELL 22 — MOV Pass/Fail
# ============================================================

def mov_flag(row):

    if row["final_suggestion"] <= 0:
        return "N/A"

    criteria = str(
        row["minimum_order_criteria"]
    ).upper()

    mov = row["current_vendor_mov"]

    if criteria == "VALUE":
        return (
            "PASS"
            if row["final_value"] >= mov
            else "FAIL"
        )

    if criteria == "CASES":
        return (
            "PASS"
            if row["final_cases_suggestion"] >= mov
            else "FAIL"
        )

    if criteria == "TONNAGE":
        return (
            "PASS"
            if row["final_tonnage"] * 1000 >= mov
            else "FAIL"
        )

    return "UNKNOWN"

In [399]:
# ============================================================
# CELL 23 — Apply MOV Check
# ============================================================

df["mov_check"] = df.apply(
    mov_flag,
    axis=1
)

df[[
    "jpin",
    "minimum_order_criteria",
    "current_vendor_mov",
    "final_suggestion",
    "final_value",
    "final_cases_suggestion",
    "final_tonnage",
    "mov_check"
]].head(20)

,jpin,minimum_order_criteria,current_vendor_mov,final_suggestion,final_value,final_cases_suggestion,final_tonnage,mov_check
0,SKU-00786,VALUE,120000,0,0.0,0,0.0000,N/A
1,SKU-00726,VALUE,30000,0,0.0,0,0.0000,N/A
2,SKU-00846,VALUE,30000,60,6551.4,1,0.0073,FAIL
3,SKU-00352,VALUE,30000,0,0.0,0,0.0000,N/A
4,SKU-00725,VALUE,30000,0,0.0,0,0.0000,N/A
5,SKU-00780,VALUE,30000,0,0.0,0,0.0000,N/A
6,SKU-00781,VALUE,30000,0,0.0,0,0.0000,N/A
7,SKU-00045,VALUE,30000,0,0.0,0,0.0000,N/A
8,SKU-00374,VALUE,30000,0,0.0,0,0.0000,N/A
9,SKU-00500,VALUE,30000,0,0.0,0,0.0000,N/A


In [400]:
# ============================================================
# CELL 24 — View final output columns
# ============================================================

df[
    [
        "jpin",
        "final_suggestion",
        "final_cases_suggestion",
        "final_value",
        "final_days_of_inventory",
        "final_tonnage",
        "mov_check",
    ]
].head(20)

,jpin,final_suggestion,final_cases_suggestion,final_value,final_days_of_inventory,final_tonnage,mov_check
0,SKU-00786,0,0,0.0,0,0.0000,N/A
1,SKU-00726,0,0,0.0,23,0.0000,N/A
2,SKU-00846,60,1,6551.4,46,0.0073,FAIL
3,SKU-00352,0,0,0.0,8,0.0000,N/A
4,SKU-00725,0,0,0.0,19,0.0000,N/A
5,SKU-00780,0,0,0.0,6,0.0000,N/A
6,SKU-00781,0,0,0.0,4,0.0000,N/A
7,SKU-00045,0,0,0.0,13,0.0000,N/A
8,SKU-00374,0,0,0.0,2,0.0000,N/A
9,SKU-00500,0,0,0.0,41,0.0000,N/A


In [401]:
# ============================================================
# CELL 25 — Save the output file
# ============================================================

output_file = "final_replenishment_data.csv"

df.to_csv(
    output_file,
    index=False
)

print(f"Output saved as: {output_file}")

Output saved as: final_replenishment_data.csv


In [402]:
# ============================================================
# CELL 26 — Download the ouput file
# ============================================================
files.download("final_replenishment_data.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [403]:
# ============================================================
# CELL 27 — remove old db if it exists
# ============================================================

db_path = Path(DB_FILE)

if db_path.exists():
    db_path.unlink()

conn = sqlite3.connect(DB_FILE)

print("SQLite database created:", DB_FILE)

SQLite database created: replenishment.db


In [404]:
# ============================================================
# CELL 28 — Database Actions
# ============================================================

base_df = pd.read_csv(output_file)

# Convert dates to strings for SQLite
if "earliest_promise_date" in base_df.columns:
    base_df["earliest_promise_date"] = pd.to_datetime(
        base_df["earliest_promise_date"],
        errors="coerce"
    ).dt.strftime("%Y-%m-%d")

# Create the table from the base CSV
base_df.to_sql(
    "replenishment_data",
    conn,
    if_exists="replace",
    index=False
)

print("Base data loaded into SQLite.")

result = pd.read_sql(
    "SELECT COUNT(*) AS row_count FROM replenishment_data",
    conn
)

display(result)

Base data loaded into SQLite.


/tmp/ipykernel_468/4285732290.py:9: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  base_df["earliest_promise_date"] = pd.to_datetime(


,row_count
0,1389


In [405]:
# ============================================================
# CELL 29 — Database actions
# ============================================================


calculated_columns = {
    "final_suggestion": "INTEGER",
    "final_cases_suggestion": "INTEGER",
    "final_value": "REAL",
    "final_days_of_inventory": "REAL",
    "final_tonnage": "REAL",
    "mov_check": "TEXT",

    # Audit / bonus fields
    "current_days_of_inventory": "REAL",
    "open_po_within_lead_time": "REAL",
    "projected_inventory": "REAL",
    "inventory_priority": "TEXT"
}

existing_columns = pd.read_sql(
    "PRAGMA table_info(replenishment_data)",
    conn
)["name"].tolist()

for column, data_type in calculated_columns.items():

    if column not in existing_columns:

        conn.execute(
            f"""
            ALTER TABLE replenishment_data
            ADD COLUMN {column} {data_type}
            """
        )

conn.commit()

print("Calculated columns added.")

Calculated columns added.


In [406]:
# ============================================================
# CELL 30 — Verify database
# ============================================================

verification_query = """
SELECT
    jpin,
    title,
    max_drr,
    current_inventory,
    final_suggestion,
    final_cases_suggestion,
    final_value,
    final_days_of_inventory,
    mov_check
FROM replenishment_data
LIMIT 10
"""

verification_df = pd.read_sql(
    verification_query,
    conn
)

display(verification_df)

,jpin,title,max_drr,current_inventory,final_suggestion,final_cases_suggestion,final_value,final_days_of_inventory,mov_check
0,SKU-00786,"Colgate Super Junior Toothbrush, Ultra Soft, 2...",0,0,0,0,0.0,0,None
1,SKU-00726,"Dabur Lal Tail, 50ml Box",6,22,0,0,0.0,23,None
2,SKU-00846,"Dabur Lal Tail, 100ml Box",2,32,60,1,6551.4,46,FAIL
3,SKU-00352,"Real Koolerz Fruit Drink, Mango, 10Rs Tetra Pak",118,0,0,0,0.0,8,None
4,SKU-00725,"Real Fruit Power Drink, Guava, 180ml Tetra Pak",35,467,0,0,0.0,19,None
5,SKU-00780,"Real Fruit Power Drink, Mixed Fruit, 180ml Tet...",154,179,0,0,0.0,6,None
6,SKU-00781,"Real Fruit Power Drink, Orange, 180ml Tetra Pa...",156,15,0,0,0.0,4,None
7,SKU-00045,"Hajmola Anardana Digestive Tablets, 40g Pack (...",23,1,0,0,0.0,13,None
8,SKU-00374,"Hajmola Regular Digestive Tablets, 20Pcs Sachet",43,9,0,0,0.0,2,None
9,SKU-00500,"Hajmola Regular Digestive Tablets, 50Pcs Bottle",5,63,0,0,0.0,41,None


In [407]:
# ============================================================
# CELL 31 — Export final CSV
# ============================================================

df.to_csv(
    output_file,
    index=False
)

print(f"Output created: {output_file}")
print("Rows:", len(df))

Output created: final_replenishment_data.csv
Rows: 1389


# Store output file in `replenishment.db`

In [408]:
final_output_df = pd.read_csv(output_file)

print(f"Loaded final output CSV: {output_file}")
print(f"Rows: {len(final_output_df)}")
display(final_output_df.head())

Loaded final output CSV: final_replenishment_data.csv
Rows: 1389


,facility_id,category_name,manufacturername,facility_name,jpin,title,pvname,vendor_lead_time,inv_norm,safety_stock,...,current_days_raw,inventory_position,target_days,target_units,raw_replenishment,residual_capacity,max_order_cases,desired_cases,mov_required_cases,projected_inventory
0,FC-001,Baby Care,Colgate-Palmolive,FC-Alpha,SKU-00786,"Colgate Super Junior Toothbrush, Ultra Soft, 2...",Baby Toothbrush,5,26,4,...,0.000000,24,30,0,0,24,1,0,0,24
1,FC-001,Baby Care,Dabur India Limited,FC-Alpha,SKU-00726,"Dabur Lal Tail, 50ml Box",Baby Oil,5,9,4,...,3.666667,142,13,78,0,98,0,0,5,142
2,FC-001,Baby Care,Dabur India Limited,FC-Alpha,SKU-00846,"Dabur Lal Tail, 100ml Box",Baby Oil,5,9,4,...,16.000000,32,13,26,0,88,1,0,5,92
3,FC-001,Beverages,Dabur India Limited,FC-Alpha,SKU-00352,"Real Koolerz Fruit Drink, Mango, 10Rs Tetra Pak",Fruit Drinks,5,14,4,...,0.000000,1000,18,2124,1124,0,0,29,109,1000
4,FC-001,Beverages,Dabur India Limited,FC-Alpha,SKU-00725,"Real Fruit Power Drink, Guava, 180ml Tetra Pak",Fruit Drinks,5,14,4,...,13.342857,677,18,630,0,0,0,0,60,677


In [409]:
# Write the final output DataFrame to a new table in the database
final_output_df.to_sql(
    "final_replenishment_data",
    conn,
    if_exists="replace",
    index=False
)

print("Final output data loaded into SQLite table 'final_replenishment_data'.")

Final output data loaded into SQLite table 'final_replenishment_data'.


In [410]:
# Verify the data has been loaded correctly
verification_final_query = "SELECT COUNT(*) AS row_count FROM final_replenishment_data"
verification_final_df = pd.read_sql(verification_final_query, conn)
display(verification_final_df)

,row_count
0,1389


In [411]:
verification_final_query_head = "SELECT * FROM final_replenishment_data LIMIT 5"
verification_final_df_head = pd.read_sql(verification_final_query_head, conn)
display(verification_final_df_head)

,facility_id,category_name,manufacturername,facility_name,jpin,title,pvname,vendor_lead_time,inv_norm,safety_stock,...,current_days_raw,inventory_position,target_days,target_units,raw_replenishment,residual_capacity,max_order_cases,desired_cases,mov_required_cases,projected_inventory
0,FC-001,Baby Care,Colgate-Palmolive,FC-Alpha,SKU-00786,"Colgate Super Junior Toothbrush, Ultra Soft, 2...",Baby Toothbrush,5,26,4,...,0.000000,24,30,0,0,24,1,0,0,24
1,FC-001,Baby Care,Dabur India Limited,FC-Alpha,SKU-00726,"Dabur Lal Tail, 50ml Box",Baby Oil,5,9,4,...,3.666667,142,13,78,0,98,0,0,5,142
2,FC-001,Baby Care,Dabur India Limited,FC-Alpha,SKU-00846,"Dabur Lal Tail, 100ml Box",Baby Oil,5,9,4,...,16.000000,32,13,26,0,88,1,0,5,92
3,FC-001,Beverages,Dabur India Limited,FC-Alpha,SKU-00352,"Real Koolerz Fruit Drink, Mango, 10Rs Tetra Pak",Fruit Drinks,5,14,4,...,0.000000,1000,18,2124,1124,0,0,29,109,1000
4,FC-001,Beverages,Dabur India Limited,FC-Alpha,SKU-00725,"Real Fruit Power Drink, Guava, 180ml Tetra Pak",Fruit Drinks,5,14,4,...,13.342857,677,18,630,0,0,0,0,60,677


In [412]:
# ============================================================
# CELL 27 — SQL QUERY 1
# Vendor-level replenishment summary
# ============================================================

query_vendor_summary = """
SELECT
    vendor_id,
    vendor_name,

    COUNT(DISTINCT jpin) AS sku_count,

    SUM(
        CASE
            WHEN final_suggestion > 0
            THEN 1
            ELSE 0
        END
    ) AS skus_requiring_replenishment,

    SUM(final_suggestion) AS total_suggested_units,

    SUM(final_cases_suggestion) AS total_suggested_cases,

    ROUND(
        SUM(final_value),
        2
    ) AS total_suggested_value,

    ROUND(
        SUM(final_tonnage),
        3
    ) AS total_suggested_tonnage,

    SUM(
        CASE
            WHEN mov_check = 'PASS'
            THEN 1
            ELSE 0
        END
    ) AS mov_pass_count,

    SUM(
        CASE
            WHEN mov_check = 'FAIL'
            THEN 1
            ELSE 0
        END
    ) AS mov_fail_count

FROM replenishment_data

GROUP BY
    vendor_id,
    vendor_name

ORDER BY
    total_suggested_value DESC;
"""

vendor_summary = pd.read_sql(
    query_vendor_summary,
    conn
)

display(vendor_summary)

,vendor_id,vendor_name,sku_count,skus_requiring_replenishment,total_suggested_units,total_suggested_cases,total_suggested_value,total_suggested_tonnage,mov_pass_count,mov_fail_count
0,VND-011,SHRI KRISHNA MURARI ENTERPRISES - Region Epsilon,25,14,18006,77,179589.00,0.789,0,14
1,VND-056,GODREJ CONSUMER PRODUCTS LTD - Region Beta,27,22,16584,74,177427.68,0.659,0,22
2,VND-038,RECKITT BENCKISER INDIA Private Limited - Regi...,36,12,3494,45,139873.28,0.584,0,12
3,VND-042,GALAXY ENTERPRISES - Region Alpha,23,10,1392,10,118823.28,0.080,0,10
4,VND-048,Nav Durga Enterprises - Region Epsilon,19,13,3230,13,106491.20,0.193,0,13
...,...,...,...,...,...,...,...,...,...,...
66,VND-049,Shiv Traders - Region Epsilon,28,0,0,0,0.00,0.000,0,0
67,VND-050,SHREE GANESH ENTERPRISES (BH_PT),6,0,0,0,0.00,0.000,0,0
68,VND-061,M/s Gupta Enterprises - Region Epsilon,16,0,0,0,0.00,0.000,0,0
69,VND-065,M/S INDO GLOBE LINKERS - Region Epsilon,6,0,0,0,0.00,0.000,0,0


In [413]:
# ============================================================
# CELL 28 — SQL QUERY 2
# Top 10 riskiest SKUs
# ============================================================

query_riskiest_skus = """
SELECT
    facility_id,
    facility_name,

    vendor_id,
    vendor_name,

    jpin,
    title,

    max_drr,
    current_inventory,

    ROUND(
        CAST(current_inventory AS REAL)
        / NULLIF(max_drr, 0),
        2
    ) AS current_days_of_inventory,

    inv_norm,
    safety_stock,

    final_suggestion,
    final_days_of_inventory,

    inventory_priority

FROM replenishment_data

WHERE
    max_drr > 0

ORDER BY
    current_days_of_inventory ASC

LIMIT 10;
"""

riskiest_skus = pd.read_sql(
    query_riskiest_skus,
    conn
)

display(riskiest_skus)

,facility_id,facility_name,vendor_id,vendor_name,jpin,title,max_drr,current_inventory,current_days_of_inventory,inv_norm,safety_stock,final_suggestion,final_days_of_inventory,inventory_priority
0,FC-001,FC-Alpha,VND-054,DABUR INDIA LIMITED - Region Epsilon,SKU-00352,"Real Koolerz Fruit Drink, Mango, 10Rs Tetra Pak",118,0,0.0,14,4,0,8,None
1,FC-001,FC-Alpha,VND-044,SIDDHI VINAYAK TRADERS - Region Alpha,SKU-00939,"Cipla Prolyte ORS, Mixed Fruit, 200ml Tetra Pak",4,0,0.0,7,4,300,75,None
2,FC-001,FC-Alpha,VND-044,SIDDHI VINAYAK TRADERS - Region Alpha,SKU-00940,"Cipla Prolyte ORS, Nimbu Paani, 200ml Tetra Pak",6,0,0.0,7,4,300,50,None
3,FC-001,FC-Alpha,VND-044,SIDDHI VINAYAK TRADERS - Region Alpha,SKU-00942,"Cipla Prolyte ORS, Apple, 200ml Tetra Pak",50,0,0.0,30,4,480,9,None
4,FC-001,FC-Alpha,VND-044,SIDDHI VINAYAK TRADERS - Region Alpha,SKU-00231,"Campa Soft Drink, Orange, 500ml Bottle",72,0,0.0,8,4,0,6,None
5,FC-001,FC-Alpha,VND-044,SIDDHI VINAYAK TRADERS - Region Alpha,SKU-00233,"Campa Soft Drink, Cola, 200ml Bottle",276,0,0.0,8,4,0,3,None
6,FC-001,FC-Alpha,VND-046,Shree Krishna Enterprises - Region Epsilon,SKU-00233,"Campa Soft Drink, Cola, 200ml Bottle",276,0,0.0,8,4,0,3,None
7,FC-001,FC-Alpha,VND-044,SIDDHI VINAYAK TRADERS - Region Alpha,SKU-00234,"Campa Soft Drink, Orange, 200ml Bottle",144,0,0.0,8,4,0,4,None
8,FC-001,FC-Alpha,VND-046,Shree Krishna Enterprises - Region Epsilon,SKU-00234,"Campa Soft Drink, Orange, 200ml Bottle",144,0,0.0,8,4,0,4,None
9,FC-001,FC-Alpha,VND-029,SLMG BEVERAGES PRIVATE LIMITED - Region Alpha,SKU-00277,"Limca Soft Drink, 2L Bottle",43,0,0.0,15,4,0,5,None
